In [0]:
# Módulos da biblioteca padrão do Python
import json
import logging
import re
import time
import warnings
from datetime import date, datetime, timedelta
from string import Template
from typing import Any, Dict, List, Tuple
from unidecode import unidecode
from pyspark.sql import DataFrame, Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.utils import AnalysisException
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    coalesce, col, concat_ws, count, current_date, current_timestamp,
    date_format, date_sub, length, lit, regexp_replace, sha2,
    to_timestamp, trim, udf, upper, when
)
from pyspark.sql.types import (
    DataType, DecimalType, IntegerType, StringType, StructField,
    StructType, TimestampType
)

In [0]:
# Realiza configuracao de logging, storage e token
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logging.getLogger("pyspark").setLevel(logging.WARNING)
logger = logging.getLogger("compass.silver")

storage_account_name = "compassdataprod"
container = "sa-compasslake"
secret_scope_name = "adlsscpkeydata"
secret_key_name   = "adlsstoragekeydata"

try:
    # Recupera o SAS Token do secret scope
    sas_token = dbutils.secrets.get(scope=secret_scope_name, key=secret_key_name)
except Exception as e:
    logger.error(f"Erro ao recuperar SAS Token do secret scope '{secret_scope_name}/{secret_key_name}': {e}")
    raise # Re-levanta a exceção para parar a execução se a credencial for crítica

# Configuração com SAS Token
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "SAS")
spark.conf.set(f"fs.azure.sas.token.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set(f"fs.azure.sas.fixed.token.{storage_account_name}.dfs.core.windows.net", sas_token)
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")



In [0]:
# A classe ExecutionMetricsCollector não realiza nenhuma operação pesada no cluster. 
# Como? => Seu único papel é consolidar metadados, como o tempo de execução, e registrar os resultados das ações do Spark que já foram executadas. As operações que demandam maior processamento — como extração, padronização, carga e validação de dados — ocorrem de forma sequencial, disparando seus próprios jobs no cluster. Entre essas operações, a função validate_data() é a que mais impacta a performance, pois envolve múltiplas contagens e filtragens de registros que é executada => após a carga dos dados na tabela Bronze.
class ExecutionMetricsCollector:
    """
    Coleta métricas de execução Spark 
    """

    def __init__(self, spark: SparkSession):
        self.spark = spark
        self.start_time = None
        self.end_time = None
        self.valid_count = 0
        self.invalid_count = 0

    def start_collection(self):
        """Marca início do processo"""
        self.start_time = datetime.now()

    def end_collection(self):
        """Marca fim do processo"""
        self.end_time = datetime.now()

    def set_counts(self, valid_count: int, invalid_count: int):
        """Define as contagens coletadas da execução da validação."""
        self.valid_count = valid_count
        self.invalid_count = invalid_count

    def collect_metrics(
        self,
        validation_results: dict,
        id_app: str,
        layer_lake: str
    ) -> str:
        """
        Gera JSON com métricas da execução.
        """
        if not self.start_time or not self.end_time:
            raise ValueError("start_collection() e end_collection() precisam ser chamados.")

        # Tempo de execução
        total_time = (self.end_time - self.start_time).total_seconds()
        formatted_time = f"{total_time:.2f} s"

        # Contagens (agora obtidas de set_counts)
        count_valid = self.valid_count
        count_invalid = self.invalid_count
        total_records = count_valid + count_invalid
        percentage_valid = (count_valid / total_records * 100) if total_records > 0 else 0.0

        # Monta dicionário de métricas (GENÉRICO)
        metrics = {
            "owner": {
                "dominio": "DOMIMIO_FICT",
                "projeto": "compass",
                "layer_lake": f"{layer_lake}"
            },
            "valid_data": {"count": count_valid, "percentage": percentage_valid},
            "invalid_data": {
                "count": count_invalid,
                "percentage": (count_invalid / total_records * 100) if total_records > 0 else 0.0,
            },
            "total_records": total_records,
            "total_processing_time": formatted_time,
            "validation_results": validation_results,
            "success_count": sum(1 for v in validation_results.values() if isinstance(v, dict) and v.get("status")),
            "error_count": sum(1 for v in validation_results.values() if isinstance(v, dict) and not v.get("status")),
            "_ts": {
                "compass_start_ts": self.start_time.strftime("%Y-%m-%d %H:%M:%S"),
                "compass_end_ts": self.end_time.strftime("%Y-%m-%d %H:%M:%S"),
            },
            "timestamp": datetime.now().strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z",
            "app_id": id_app,
        }

        return json.dumps(metrics, indent=2)

In [0]:
# ===================== Variaveis de entrada via param
date_partition = dbutils.widgets.get("date_partition")
# date_partition       = "2025-09-14"

table_target         = "g_compass.reviews_customer_compass"
target_mode          = "overwrite"
target_mode_table    = "delta"

params = {
    "date_partition": date_partition
}

logging.info("Parâmetros de entrada: %s", json.dumps(params))

In [0]:
# ===================== Função de leitura
def func_read_data(
    spark: SparkSession,
    table_name: str,
    date_partition: str
) -> DataFrame:
    """
    Lê dados de uma tabela Delta filtrando por date_load no formato yyyyMM
    e adiciona coluna de auditoria.

    Args:
        spark (SparkSession): Sessão Spark.
        table_name (str): Nome da tabela no catálogo.
        date_partition (str): Data de referência (yyyy-MM-dd).

    Returns:
        DataFrame: DataFrame filtrado e padronizado.

    Raises:
        ValueError: Se a coluna 'date_load' não existir.
    """
    try:
        # Extrair yyyyMM da data de entrada
        yyyymm_ref = date_partition[:7].replace("-", "")
        logger.info(f"Lendo tabela '{table_name}' filtrando date_load em yyyyMM={yyyymm_ref}")

        # Ler tabela
        df = spark.read.table(table_name)
        current_columns = df.columns

        # Validar existência da coluna obrigatória
        if "date_load" not in current_columns:
            raise ValueError(f"Tabela '{table_name}' não contém a coluna obrigatória 'date_load'.")

        # Filtrar pelo yyyyMM da coluna date_load
        df = (
            df.filter(col("date_load") == yyyymm_ref)
       )

        return df

    except AnalysisException as e:
        logger.error(f"Erro ao ler a tabela '{table_name}': {e}")
        raise

df = func_read_data(
    spark,
    table_name="s_compass.instituicao_reviews",
    date_partition=date_partition
)


In [0]:
def prepare_data(df: DataFrame) -> DataFrame:
    """
    Seleciona, trata e padroniza colunas em um DataFrame do Spark.

    Args:
        df: O DataFrame de entrada com dados de reviews do Spark.

    Returns:
        Um novo DataFrame tratado e com as colunas necessárias para agregação.
    """
    # 1. Seleciona apenas as colunas relevantes.
    # 2. Converte a coluna `review_date` para um tipo de data adequado e, em seguida,
    #    a trunca para o início do mês. Isso garante que a data esteja no formato
    #    correto para a agregação mensal no seu dashboard.
    df_prepared = (
        df.select(
            'review_date',
            'review_rating',
            'review_version',
            'segment',
            'service_type',
            'app_reference'
        )
        .withColumn(
            'review_date',
            F.to_timestamp(F.col('review_date'), 'yyyy-MM-dd\'T\'HH:mm:ss.SSSZ')
        )
        .withColumn(
            'review_year',
            F.date_format(F.col('review_date'), 'yyyy')
        )
        .withColumn(
            'review_month',
            F.date_format(F.col('review_date'), 'MM')
        )
    )

    # Remove linhas com valores nulos (NaN) nas colunas críticas usadas para a agregação.
    df_prepared = df_prepared.na.drop(subset=['review_date', 'review_rating'])

    return df_prepared


prepared_df = prepare_data(df)

In [0]:
def aggregate_reviews(df: DataFrame) -> DataFrame:
    """
    Agrega reviews por data, nota, versão, segmento, tipo de serviço e app,
    calculando a contagem de reviews e a média das avaliações por grupo.

    => average_rating: A média das notas de todas as avaliações do grupo.
    => min_rating: A menor nota encontrada no grupo.
    => max_rating: A maior nota encontrada no grupo.
    => nps_score: O que é Net Promoter Score? É uma métrica de negócio crucial para a experiência do cliente. Ele é calculado com base nos promotores (nota 5) e detratores (notas de 1 a 3) do período.



    Args:
        df: O DataFrame de entrada, que deve ter sido preparado pela função prepare_data.

    Returns:
        Um novo DataFrame com a contagem de reviews e a média das avaliações por grupo.
    """
    # Agrupa pelos campos solicitados e conta o número de reviews e
    # calcula a média das notas em cada grupo.
    df_aggregated = (
        df.groupBy(
            'review_year',
            'review_month',
            'review_version',
            'segment',
            'service_type',
            'app_reference'
        )
        .agg(
            F.count('*').alias('review_count'),
            # Calcula a média e explicitamente converte o resultado para o tipo Double.
            F.round(F.avg('review_rating'), 1).cast('double').alias('average_rating'),
            # Adiciona o cálculo da nota mínima do grupo
            F.min('review_rating').cast('double').alias('min_rating'),
            # Adiciona o cálculo da nota máxima do grupo
            F.max('review_rating').cast('double').alias('max_rating'),
            # Adiciona o cálculo do NPS, que compara promotores (rating 5) com detratores (rating <= 3)
            F.round(
                (
                    (F.sum(F.when(F.col('review_rating') == 5, 1).otherwise(0)))
                    -
                    (F.sum(F.when(F.col('review_rating') <= 3, 1).otherwise(0)))
                )
                / F.count('*'),
                2
            ).cast('double').alias('nps_score')
        )
    )

    return df_aggregated


aggregated_df = aggregate_reviews(prepared_df)

display(aggregated_df)

In [0]:
def save_data(
    df: DataFrame,
    table_name: str,
    partition_col: str = None,
    target_format: str = "delta"
):
    """
    Grava um DataFrame como tabela Delta particionada, sempre usando overwrite.

    Args:
        df: DataFrame do Spark
        table_name: nome da tabela Delta
        partition_col: coluna para particionamento (ex: "review_year")
        target_format: formato de saída (default "delta")
    """
    logger.info("Iniciando gravação da tabela...")

    if df.rdd.isEmpty():
        logger.warning("Nenhum dado para gravar. Tabela não será atualizada.")
        return

    writer = df.write.format(target_format).mode("overwrite")

    if partition_col:
        writer = writer.partitionBy(partition_col)

    # Sobrescreve o schema da tabela caso tenha alterado alguma coluna
    writer = writer.option("overwriteSchema", "true")

    writer.saveAsTable(table_name)
    logger.info(f"Tabela {table_name} gravada com sucesso no formato {target_format}.")


save_data(
    df=aggregated_df,
    table_name=table_target,
    partition_col="review_year",
    target_format="delta",
)


logger.info("Processo de ingestão finalizado, iniciando o trabalho de coleta de métricas!")